# Ocupação de UTI e estimação de parâmetros hospitalares por mínimos quadrados

Notebook para o Google Colab com duas análises, sobre os dados do repositório
*Modelagem-SEIRS-para-simular-evolucao-da-COVID-19* (Brasil, 2007–2026):

1. **Ocupação dos leitos de UTI SUS** (séries mensais, sazonalidade, UFs, tendência).
2. **Estimação por mínimos quadrados** dos parâmetros hospitalares (taxas σ e γ de saída de cada
   compartimento e probabilidades p_H, p_C, d, d_H), comparada às estimativas por média do repositório.

> **Escopo dos dados.** As bases são de **sarampo** (CID-10 B05: SINAN e SIH), não de COVID-19.
> Já a **ocupação de UTI** (SIH + CNES) é de **todas as causas**, não só do sarampo.

**Como usar:** *Ambiente de execução → Executar tudo*. Na célula [2.1], selecione de uma vez os CSVs
listados abaixo (cópias de `dados/` e `parametros/` do repositório, sem compactar).

| Arquivos a enviar | Quantidade |
|---|---|
| `notificacoes_exantematicas_sinan_2007.csv` … `_2026.csv` (`dados/sinan/`) | 20 |
| `internacoes_sarampo_sih_2007.csv` … `_2026.csv` (`dados/sih/`) | 20 |
| `ocupacao_uti_brasil.csv`, `ocupacao_uti_mensal_uf.csv` (`dados/resultados/hospitalar/`) | 2 |
| `leitos_uti_brasil.csv`, `leitos_uti_mensal_uf.csv` (`dados/resultados/leitos/`) | 2 |

Os demais CSVs do repositório (`leitos_cnes_*`, `sih_uti_*`, `populacao_*`) não são necessários: já
estão refletidos nas séries de ocupação e de leitos.

| Parte | Conteúdo |
|---|---|
| 1 | Ambiente |
| 2 | Dados: envio, validação e carga |
| 3 | Análise da ocupação de UTI |
| 4 | Estimação por mínimos quadrados |
| 5 | Exportar resultados |

## Parte 1 — Ambiente
### [1.1] Bibliotecas e pastas
O Colab já traz todas as bibliotecas. As pastas replicam a árvore do repositório (`dados/` ao lado de
`parametros/`), que o script de leitura da Parte 2 espera.

In [ ]:
# [1.1] Bibliotecas e pastas
import os, sys, shutil
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import optimize, stats

BASE = Path("/content") if Path("/content").exists() else Path("colab_run").resolve()
for sub in ("dados/sinan", "dados/sih", "dados/resultados/leitos", "dados/resultados/hospitalar"):
    (BASE / sub).mkdir(parents=True, exist_ok=True)
os.chdir(BASE)
sys.path.insert(0, str(BASE / "parametros"))

pd.set_option("display.width", 200, "display.max_colwidth", 80, "display.float_format", "{:,.4f}".format)
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": .3})
print("python", sys.version.split()[0], "| pandas", pd.__version__, "| numpy", np.__version__)
print("Pasta base:", BASE)

## Parte 2 — Dados
### [2.1] Envio dos CSVs
Abre a janela de upload do Colab. Selecione **todos** os CSVs da tabela da introdução de uma vez
(Ctrl+A na pasta). Cada arquivo é guardado na pasta correspondente pelo prefixo do nome.

*Fora do Colab*, defina a variável de ambiente `REPO_DIR` com a raiz do repositório e os arquivos são copiados de lá.

In [ ]:
# [2.1] Upload dos CSVs (ou copia local via REPO_DIR)
DESTINOS = {                       # prefixo do arquivo -> pasta
    "notificacoes_exantematicas_sinan_": "dados/sinan",
    "internacoes_sarampo_sih_":         "dados/sih",
    "leitos_uti_":          "dados/resultados/leitos",
    "ocupacao_uti_":        "parametros",
}

def destino(nome):
    for prefixo, pasta in DESTINOS.items():
        if nome.startswith(prefixo):
            return BASE / pasta / nome
    return None

try:
    from google.colab import files
    enviados = files.upload()                       # {nome: bytes}
    for nome, conteudo in enviados.items():
        alvo = destino(nome)
        if alvo is None:
            print("ignorado (nao esperado):", nome)
            continue
        alvo.write_bytes(conteudo)
        (BASE / nome).unlink(missing_ok=True)       # remove a copia solta em /content
except ImportError:
    repo = Path(os.environ["REPO_DIR"])
    for arq in list(repo.glob("dados/**/*.csv")):
        alvo = destino(arq.name)
        if alvo is not None:
            shutil.copy(arq, alvo)
print("Envio concluido.")

### [2.2] Validação
Confere se todos os arquivos esperados chegaram e mostra linhas e período de cada base.

In [ ]:
# [2.2] Validar arquivos enviados
ANOS_ESPERADOS = range(2007, 2027)
esperados = (
    [BASE / f"dados/sinan/notificacoes_exantematicas_sinan_{a}.csv" for a in ANOS_ESPERADOS]
    + [BASE / f"dados/sih/internacoes_sarampo_sih_{a}.csv" for a in ANOS_ESPERADOS]
    + [BASE / "dados/resultados/hospitalar/ocupacao_uti_brasil.csv", BASE / "dados/resultados/hospitalar/ocupacao_uti_mensal_uf.csv",
       BASE / "dados/resultados/leitos/leitos_uti_brasil.csv", BASE / "dados/resultados/leitos/leitos_uti_mensal_uf.csv"]
)
faltam = [p.name for p in esperados if not p.exists()]
assert not faltam, f"Faltam {len(faltam)} arquivo(s): {faltam}"

linhas = []
for nome, padrao in [("SINAN", "dados/sinan/*.csv"), ("SIH", "dados/sih/*.csv")]:
    arqs = sorted(BASE.glob(padrao))
    n = sum(sum(1 for _ in open(a, encoding="utf-8")) - 1 for a in arqs)
    linhas.append({"base": nome, "arquivos": len(arqs), "linhas": n,
                   "periodo": f"{arqs[0].stem[-4:]}-{arqs[-1].stem[-4:]}"})
pd.DataFrame(linhas).set_index("base")

### [2.3] Script de leitura do SINAN e do SIH
Grava, **sem alterações**, o script `calcular_parametros_hospitalares.py` do repositório. Ele é usado só
como biblioteca: `carregar_sinan` e `carregar_sih` aplicam a limpeza dos dados (casos confirmados de
sarampo, internações com CID B05, junção das AIHs de continuação) e `estimar` dá as estimativas por média,
que servem de linha de base para o mínimos quadrados.

In [ ]:
%%writefile parametros/calcular_parametros_hospitalares.py
"""
Estima os parâmetros da dinâmica de hospitalização do sarampo a partir dos
microdados do DATASUS já exportados em CSV (ver exportar_datasus.py):

    dados/sinan/notificacoes_exantematicas_sinan_AAAA.csv  ->  p_H, sigma_I1, gamma_I1
    dados/sih/internacoes_sarampo_sih_AAAA.csv            ->  p_C, sigma_H, gamma_H, d, sigma_C, gamma_C

Fluxos do modelo:
    I1 -> H : p_H * sigma_I1 * I1          I1 -> R : (1 - p_H) * gamma_I1 * I1
    H  -> C : p_C * theta * sigma_H * H    H  -> R : (1 - p_C) * gamma_H * H
    H  -> D : p_C * (1 - theta) * sigma_H * H (UTI lotada)
    C  -> D : d * sigma_C * C              C  -> R : (1 - d) * gamma_C * C

    python parametros/calcular_parametros_hospitalares.py
    python parametros/calcular_parametros_hospitalares.py --anos 2018 2019 2020 --ufs SP PR
"""

import os
import argparse

import numpy as np
import pandas as pd

# =============================================================================
# CONFIGURACOES
# =============================================================================

RAIZ            = os.path.dirname(os.path.abspath(__file__))
DIRETORIO_DADOS = os.path.join(RAIZ, "..", "dados")                  # CSVs de exportar_datasus.py
ARQUIVO_SAIDA   = os.path.join(RAIZ, "parametros_hospitalares.csv")

# Período e UFs da estimativa (padrão: todo o período exportado, Brasil inteiro)
ANOS = list(range(2007, 2027))
TODAS_UFS = ["AC", "AL", "AM", "AP", "BA", "CE", "DF", "ES", "GO", "MA", "MG",
             "MS", "MT", "PA", "PB", "PE", "PI", "PR", "RJ", "RN", "RO", "RR",
             "RS", "SC", "SE", "SP", "TO"]

# CID-10 do sarampo. Complicações (ex.: pneumonia, J18) às vezes entram como
# diagnóstico principal com B05 como secundário.
CID = "B05"
USAR_DIAG_SECUNDARIO = True

# Janela plausível entre início dos sintomas e internação, em dias. Fora dela
# o registro é tratado como erro de digitação.
DIAS_ATE_INTERNACAO = (0, 30)

# Códigos do SINAN (dicionário das doenças exantemáticas)
CLASSI_SARAMPO = "1"                                  # CLASSI_FIN: 1 = sarampo
HOSPIT_SIM, HOSPIT_NAO = "1", "2"                     # ID_HOSPIT
VACINA_SIM, VACINA_NAO = "1", "2"                     # CS_VACINA
EVOLUCAO_OBITO_AGRAVO = "2"                           # EVOLUCAO

# Pós-estratificação de p_H: os casos com ID_HOSPIT em branco/ignorado recebem a
# fração internada do seu estrato. Os brancos se concentram em 2019-2020 (SP e
# PA), então a média nacional dos casos completos não os representa bem.
# Cada caso usa o nível mais detalhado cujo estrato tenha pelo menos
# MIN_INFORMADOS_ESTRATO casos com ID_HOSPIT preenchido; senão, sobe um nível.
NIVEIS_ESTRATO = [["ANO", "UF"], ["ANO"], []]
MIN_INFORMADOS_ESTRATO = 20

Z95 = 1.959964


# =============================================================================
# 1. LEITURA DOS CSVS
# =============================================================================

def ler_csvs(base, prefixo, anos, colunas):
    """
    Concatena dados/<base>/<prefixo>_AAAA.csv dos `anos` pedidos
    (os que não existirem são ignorados), só com as `colunas` presentes. Tudo é
    lido como texto: os códigos do DATASUS têm zeros à esquerda.
    """
    partes = []
    for ano in anos:
        caminho = os.path.join(DIRETORIO_DADOS, base, f"{prefixo}_{ano}.csv")
        if not os.path.exists(caminho):
            continue
        partes.append(pd.read_csv(caminho, sep=";", dtype=str, keep_default_na=False,
                                  usecols=lambda c: c in colunas))
    if not partes:
        raise FileNotFoundError(f"nenhum CSV {base}/{prefixo}_AAAA.csv para {anos}; "
                                "rode antes parametros/exportar_datasus.py")
    return pd.concat(partes, ignore_index=True).fillna("")  # arquivos antigos têm menos colunas


# =============================================================================
# 2. SINAN: CASOS CONFIRMADOS DE SARAMPO
# =============================================================================

COLUNAS_SINAN = ["CLASSI_FIN", "DT_SIN_PRI", "DT_INTERNA", "ID_HOSPIT",
                 "CS_VACINA", "EVOLUCAO", "SG_UF", "SG_UF_NOT", "UF_H"]

def carregar_sinan(anos):
    """Casos confirmados com início dos sintomas nos `anos` pedidos."""
    # O arquivo é por ano de notificação; um caso do fim do ano pode ser
    # notificado no ano seguinte, então lê também o arquivo do ano seguinte.
    sinan = ler_csvs("sinan", "notificacoes_exantematicas_sinan", sorted(set(anos) | {max(anos) + 1}), COLUNAS_SINAN)
    print(f"  {len(sinan)} notificações")
    sinan = sinan[sinan["CLASSI_FIN"] == CLASSI_SARAMPO].copy()
    for col in ["DT_SIN_PRI", "DT_INTERNA"]:
        sinan[col] = pd.to_datetime(sinan[col], format="%Y%m%d", errors="coerce")
    sinan = sinan[sinan["DT_SIN_PRI"].dt.year.isin(anos)].copy()

    # Dedução lógica: hospitalização em branco/ignorada, mas com a UF do
    # hospital preenchida, só pode ser internação.
    sem_info = ~sinan["ID_HOSPIT"].isin([HOSPIT_SIM, HOSPIT_NAO])
    com_hospital = sinan["UF_H"] != ""
    sinan.loc[sem_info & com_hospital, "ID_HOSPIT"] = HOSPIT_SIM
    print(f"  {(sem_info & com_hospital).sum()} casos com ID_HOSPIT deduzido pela UF do hospital")

    # Variáveis dos estratos (UF de residência; na falta, a de notificação)
    sinan["ANO"] = sinan["DT_SIN_PRI"].dt.year
    sinan["UF"] = sinan["SG_UF"].where(sinan["SG_UF"] != "", sinan["SG_UF_NOT"])
    return sinan


# =============================================================================
# 3. SIH: INTERNACOES COM CID DE SARAMPO
# =============================================================================

COLUNAS_DIAG = ["DIAG_PRINC", "DIAG_SECUN"] + [f"DIAGSEC{k}" for k in range(1, 10)]
COLUNAS_SIH = ["UF", "COMPETEN", "N_AIH", "IDENT", "DT_INTER", "DT_SAIDA", "DIAS_PERM",
               "UTI_MES_TO", "MARCA_UTI", "MORTE"] + COLUNAS_DIAG

def carregar_sih(anos, ufs):
    """AIHs de sarampo internadas nos `anos` pedidos, uma linha por internação."""
    # O CSV é por ano de competência (mês de processamento), que pode vir meses
    # depois da internação: lê também o ano seguinte e filtra pela data de internação.
    sih = ler_csvs("sih", "internacoes_sarampo_sih", sorted(set(anos) | {max(anos) + 1}), COLUNAS_SIH)
    for col in COLUNAS_SIH:                           # arquivos antigos não têm DIAGSEC1-9
        if col not in sih:
            sih[col] = ""
    sih = sih[sih["UF"].isin(ufs)].copy()

    # O CSV traz o CID em qualquer diagnóstico; aqui se decide se os secundários contam
    diags = ["DIAG_PRINC"] + (COLUNAS_DIAG[1:] if USAR_DIAG_SECUNDARIO else [])
    sih = sih[np.logical_or.reduce([sih[c].str.startswith(CID) for c in diags])].copy()
    print(f"  {len(sih)} AIHs com {CID}")

    for col in ["DIAS_PERM", "UTI_MES_TO", "MORTE"]:
        sih[col] = pd.to_numeric(sih[col], errors="coerce").fillna(0).astype(int)
    for col in ["DT_INTER", "DT_SAIDA"]:
        sih[col] = pd.to_datetime(sih[col], format="%Y%m%d", errors="coerce")
    sih["CID_PRINCIPAL"] = sih["DIAG_PRINC"].str.startswith(CID)

    # Internações longas geram AIHs de continuação (IDENT = 5) com o mesmo
    # número: soma os dias e junta num único registro por internação.
    internacoes = (sih.groupby(["UF", "N_AIH"], as_index=False)
                      .agg(DT_INTER=("DT_INTER", "min"),
                           DT_SAIDA=("DT_SAIDA", "max"),
                           DIAS_PERM=("DIAS_PERM", "sum"),
                           UTI_DIAS=("UTI_MES_TO", "sum"),
                           MORTE=("MORTE", "max"),
                           CID_PRINCIPAL=("CID_PRINCIPAL", "max")))
    return internacoes[internacoes["DT_INTER"].dt.year.isin(anos)]


# =============================================================================
# 4. ESTIMATIVAS
# =============================================================================

def proporcao(sucessos, total):
    """Proporção com intervalo de confiança de Wilson (95%)."""
    if total == 0:
        return np.nan, np.nan, np.nan
    p = sucessos / total
    centro = (p + Z95**2 / (2 * total)) / (1 + Z95**2 / total)
    meia = Z95 * np.sqrt(p * (1 - p) / total + Z95**2 / (4 * total**2)) / (1 + Z95**2 / total)
    return p, max(0.0, centro - meia), min(1.0, centro + meia)

def taxa(duracoes):
    """
    Taxa = 1 / duração média (1/dia). O intervalo vem do IC 95% da média,
    invertido (a média maior dá a taxa menor).
    """
    duracoes = np.asarray(duracoes, dtype=float)
    n = len(duracoes)
    if n == 0 or duracoes.mean() <= 0:
        return np.nan, np.nan, np.nan
    media = duracoes.mean()
    erro = Z95 * duracoes.std(ddof=1) / np.sqrt(n) if n > 1 else np.nan
    sup = 1 / (media - erro) if media - erro > 0 else np.inf
    return 1 / media, 1 / (media + erro), sup

def media_dias(duracoes):
    """Duração média em dias, com IC 95% (aproximação normal)."""
    duracoes = np.asarray(duracoes, dtype=float)
    n = len(duracoes)
    if n == 0:
        return np.nan, np.nan, np.nan
    media = duracoes.mean()
    erro = Z95 * duracoes.std(ddof=1) / np.sqrt(n) if n > 1 else np.nan
    return media, max(0.0, media - erro), media + erro           # duração não é negativa

def proporcao_pos_estratificada(casos, informado, sucesso):
    """
    Proporção pós-estratificada: p = soma_g (N_g / N) * p_g, em que N_g conta
    TODOS os casos do estrato g (inclusive os sem informação) e p_g vem só dos
    informados. Equivale a imputar em cada caso sem informação a proporção do
    seu estrato (hipótese MAR: dentro do estrato, a falta é aleatória).

    Cada caso usa o primeiro nível de NIVEIS_ESTRATO cujo estrato tenha ao menos
    MIN_INFORMADOS_ESTRATO informados. Variância: soma_g (N_g/N)^2 p_g(1-p_g)/n_g,
    que ignora a pequena covariância entre estratos de níveis diferentes.

    Devolve ((p, ic_inf, ic_sup), casos por nível).
    """
    casos = casos.reset_index(drop=True)
    informado = np.asarray(informado)
    sucesso = np.asarray(sucesso) & informado
    pendente = np.ones(len(casos), dtype=bool)
    celulas, por_nivel = [], {}

    for i, nivel in enumerate(NIVEIS_ESTRATO):
        ultimo = i == len(NIVEIS_ESTRATO) - 1
        chave = (casos[nivel].astype(str).agg("|".join, axis=1) if nivel
                 else pd.Series("Brasil", index=casos.index))
        estratos = pd.DataFrame({"N": pd.Series(pendente).groupby(chave).sum(),
                                 "n": pd.Series(informado).groupby(chave).sum(),
                                 "s": pd.Series(sucesso).groupby(chave).sum()})
        aceitos = estratos[(estratos["N"] > 0) & ((estratos["n"] >= MIN_INFORMADOS_ESTRATO) | ultimo)]
        celulas.append(aceitos)

        usados = pendente & chave.isin(aceitos.index).to_numpy()
        por_nivel[" x ".join(nivel) or "Brasil"] = int(usados.sum())
        pendente &= ~usados

    celulas = pd.concat(celulas)
    celulas = celulas[celulas["n"] > 0]                   # estrato sem nenhum informado não contribui
    peso = celulas["N"] / celulas["N"].sum()
    p_g = celulas["s"] / celulas["n"]
    p = (peso * p_g).sum()
    erro = Z95 * np.sqrt((peso**2 * p_g * (1 - p_g) / celulas["n"]).sum())
    return (p, max(0.0, p - erro), min(1.0, p + erro)), por_nivel

def estimar(sinan, sih):
    linhas = []

    def registrar(parametro, estimativa, n, fonte, descricao):
        valor, inf, sup = estimativa
        linhas.append(dict(parametro=parametro, valor=valor, ic95_inf=inf, ic95_sup=sup,
                           n=n, fonte=fonte, descricao=descricao))

    # --- SINAN ---------------------------------------------------------------
    hosp = sinan["ID_HOSPIT"]
    n_sim, n_nao = (hosp == HOSPIT_SIM).sum(), (hosp == HOSPIT_NAO).sum()
    registrar("p_H", proporcao(n_sim, n_sim + n_nao), n_sim + n_nao, "SINAN",
              "fração internada, entre os casos com ID_HOSPIT preenchido (sim/não)")
    registrar("p_H_minimo", proporcao(n_sim, len(sinan)), len(sinan), "SINAN",
              "limite inferior: internados / todos os confirmados (em branco = não internado)")

    for codigo, rotulo in [(VACINA_NAO, "nao_vacinados"), (VACINA_SIM, "vacinados")]:
        grupo = hosp[sinan["CS_VACINA"] == codigo]
        s, t = (grupo == HOSPIT_SIM).sum(), grupo.isin([HOSPIT_SIM, HOSPIT_NAO]).sum()
        registrar(f"p_H_{rotulo}", proporcao(s, t), t, "SINAN",
                  f"fração internada entre os {rotulo.replace('_', ' ')} (CS_VACINA)")

    informado = hosp.isin([HOSPIT_SIM, HOSPIT_NAO])
    for filtro, sufixo in [(slice(None), ""), (sinan["CS_VACINA"] == VACINA_NAO, "_nao_vacinados")]:
        grupo = sinan[filtro]
        estimativa, por_nivel = proporcao_pos_estratificada(
            grupo, informado[filtro], (hosp == HOSPIT_SIM)[filtro])
        niveis = ", ".join(f"{nome}: {n}" for nome, n in por_nivel.items())
        registrar(f"p_H{sufixo}_estratificado", estimativa, len(grupo), "SINAN",
                  f"pós-estratificado por ano x UF (casos por nível: {niveis})")

    # Tempo em I1. Só é mensurável para quem foi internado (DT_INTERNA); o SINAN
    # não tem data de cura, então o tempo até R não é observado.
    dias = (sinan["DT_INTERNA"] - sinan["DT_SIN_PRI"]).dt.days
    dias = dias[(hosp == HOSPIT_SIM) & dias.between(*DIAS_ATE_INTERNACAO)]
    registrar("dias_I1_ate_H", media_dias(dias), len(dias), "SINAN",
              "dias entre início dos sintomas e internação (observado)")
    registrar("dias_I1_ate_R", (np.nan, np.nan, np.nan), 0, "SINAN",
              "não observável: o SINAN não registra a data de cura")

    # No diagrama, I1 sai por dois fluxos: p_H*sigma_I1*I1 e (1-p_H)*gamma_I1*I1.
    # Com saídas exponenciais concorrentes, o tempo em I1 tem a mesma distribuição
    # para os dois destinos, e só a taxa total é estimável a partir das durações.
    # O estimador de máxima verossimilhança dessa taxa usa as durações observadas
    # (as dos internados); com sigma_I1 = gamma_I1 = essa taxa, p_H é a fração internada.
    registrar("sigma_I1", taxa(dias), len(dias), "SINAN",
              "taxa I1 -> H (1/dia): 1 / dias médios até a internação")
    registrar("gamma_I1", taxa(dias), len(dias), "SINAN",
              "taxa I1 -> R (1/dia): igual a sigma_I1 (tempo até R não observado; ver dias_I1_ate_R)")

    obitos = (sinan["EVOLUCAO"] == EVOLUCAO_OBITO_AGRAVO).sum()
    registrar("obitos_sinan", (obitos, np.nan, np.nan), len(sinan), "SINAN",
              "óbitos pelo agravo entre os confirmados (validação, não entra no modelo)")

    # --- SIH -----------------------------------------------------------------
    com_uti = sih["UTI_DIAS"] > 0
    obito = sih["MORTE"] == 1

    # Tempo em H (enfermaria antes da UTI ou até a alta):
    #   sem UTI           -> DIAS_PERM
    #   com UTI e óbito   -> DIAS_PERM - UTI_DIAS, supondo que o óbito ocorreu na UTI
    #                        (o SIH não informa onde)
    #   com UTI e alta    -> não separável: os dias fora da UTI podem ser antes ou
    #                        depois dela, e o SIH não registra a ordem
    dias_H_ate_R = sih.loc[~com_uti & ~obito, "DIAS_PERM"]
    dias_H_ate_C = (sih.loc[com_uti & obito, "DIAS_PERM"] - sih.loc[com_uti & obito, "UTI_DIAS"]).clip(lower=0)
    dias_H = pd.concat([sih.loc[~com_uti, "DIAS_PERM"], dias_H_ate_C])
    registrar("dias_H_ate_R", media_dias(dias_H_ate_R), len(dias_H_ate_R), "SIH",
              "dias de enfermaria das internações sem UTI com alta (observado)")
    registrar("dias_H_ate_C", media_dias(dias_H_ate_C), len(dias_H_ate_C), "SIH",
              "dias de enfermaria antes da UTI, só nos óbitos em UTI (observado)")

    # Mesma situação de I1: H sai por p_C*sigma_H*H e (1-p_C)*gamma_H*H, e só a
    # taxa total de saída é estimável a partir das durações.
    registrar("p_C", proporcao(com_uti.sum(), len(sih)), len(sih), "SIH",
              "fração das internações com dias de UTI")
    registrar("sigma_H", taxa(dias_H), len(dias_H), "SIH",
              "taxa H -> C (1/dia): 1 / dias médios em H (sem UTI + antes da UTI nos óbitos)")
    registrar("gamma_H", taxa(dias_H), len(dias_H), "SIH",
              "taxa H -> R (1/dia): igual a sigma_H (mesma taxa total de saída de H)")

    # C sai por d*sigma_C*C (óbito) e (1-d)*gamma_C*C (alta)
    dias_C = sih.loc[com_uti, "UTI_DIAS"]
    registrar("dias_C_ate_D", media_dias(sih.loc[com_uti & obito, "UTI_DIAS"]), (com_uti & obito).sum(), "SIH",
              "dias de UTI dos óbitos (observado)")
    registrar("dias_C_ate_R", media_dias(sih.loc[com_uti & ~obito, "UTI_DIAS"]), (com_uti & ~obito).sum(), "SIH",
              "dias de UTI das altas (observado)")
    registrar("d", proporcao((com_uti & obito).sum(), com_uti.sum()), com_uti.sum(), "SIH",
              "fração de óbitos entre as internações com UTI")
    registrar("sigma_C", taxa(dias_C), len(dias_C), "SIH",
              "taxa C -> D (1/dia): 1 / dias médios de UTI (óbitos + altas)")
    registrar("gamma_C", taxa(dias_C), len(dias_C), "SIH",
              "taxa C -> R (1/dia): igual a sigma_C (mesma taxa total de saída de C)")
    registrar("d_H", proporcao(sih.loc[~com_uti, "MORTE"].sum(), (~com_uti).sum()), (~com_uti).sum(), "SIH",
              "fração de óbitos entre as internações sem UTI (se relevante, criar o fluxo H -> D)")

    return pd.DataFrame(linhas)

def resumo_por_ano(sinan, sih):
    """Contagens por ano, para ver de onde vem a informação de cada parâmetro."""
    ano_sinan = sinan["DT_SIN_PRI"].dt.year
    ano_sih = sih["DT_INTER"].dt.year
    resumo = pd.DataFrame({
        "confirmados": ano_sinan.value_counts(),
        "internados_sinan": ano_sinan[sinan["ID_HOSPIT"] == HOSPIT_SIM].value_counts(),
        "obitos_sinan": ano_sinan[sinan["EVOLUCAO"] == EVOLUCAO_OBITO_AGRAVO].value_counts(),
        "internacoes_sih": ano_sih.value_counts(),
        "internacoes_uti": ano_sih[sih["UTI_DIAS"] > 0].value_counts(),
        "obitos_sih": ano_sih[sih["MORTE"] == 1].value_counts(),
    })
    resumo.index.name = "ano"
    return resumo.fillna(0).astype(int).sort_index()


# =============================================================================
# 5. EXECUCAO
# =============================================================================

def main():
    parser = argparse.ArgumentParser(description=__doc__.split("\n\n")[0])
    parser.add_argument("--anos", type=int, nargs="+", default=ANOS)
    parser.add_argument("--ufs", nargs="+", default=["TODAS"],
                        help="UFs do SIH, ou TODAS")
    args = parser.parse_args()
    ufs = TODAS_UFS if args.ufs == ["TODAS"] else [u.upper() for u in args.ufs]

    print(f"Anos: {args.anos[0]}-{args.anos[-1]}" if len(args.anos) > 3 else f"Anos: {args.anos}")
    print("Lendo SINAN (Brasil)...")
    sinan = carregar_sinan(args.anos)
    print(f"  {len(sinan)} casos confirmados de sarampo")

    print(f"Lendo SIH ({'Brasil' if ufs == TODAS_UFS else ', '.join(ufs)})...")
    sih = carregar_sih(args.anos, ufs)
    print(f"  {len(sih)} internações ({sih['CID_PRINCIPAL'].sum()} com {CID} como diagnóstico principal)")

    parametros = estimar(sinan, sih)
    parametros.to_csv(ARQUIVO_SAIDA, index=False, sep=";", encoding="utf-8-sig")
    resumo = resumo_por_ano(sinan, sih)
    resumo.to_csv(os.path.join(RAIZ, "parametros_hospitalares_por_ano.csv"), sep=";")

    pd.set_option("display.width", 200)
    pd.set_option("display.max_colwidth", 90)
    print()
    print(resumo.to_string())
    print()
    print(parametros[["parametro", "valor", "ic95_inf", "ic95_sup", "n", "fonte"]]
          .to_string(index=False, float_format=lambda v: f"{v:.4f}"))
    print(f"\nParâmetros salvos em {ARQUIVO_SAIDA}")

    preenchidos = sinan["ID_HOSPIT"].isin([HOSPIT_SIM, HOSPIT_NAO]).mean()
    if preenchidos < 0.9:
        print(f"\nAtenção: só {preenchidos:.0%} dos confirmados têm ID_HOSPIT preenchido; "
              "p_H pode estar superestimado (casos graves tendem a ser mais bem investigados). "
              "Use p_H_minimo como limite inferior.")
    n_uti = (sih["UTI_DIAS"] > 0).sum()
    if n_uti < 30:
        print(f"Atenção: só {n_uti} internações com UTI; nu_C e d ficam instáveis. "
              "Considere valores da literatura.")


if __name__ == "__main__":
    main()

### [2.4] Carga dos dados
Carrega os dados prontos para análise:

- `sinan`: casos confirmados de sarampo (uma linha por caso);
- `sih`: internações por sarampo (uma linha por internação);
- `ocup`, `ocup_uf`: ocupação de UTI SUS, Brasil e por UF (mensal);
- `leitos`: leitos de UTI por tipo e por 10 mil habitantes (mensal, Brasil).

In [ ]:
# [2.4] Carregar dados
import calcular_parametros_hospitalares as cph

ANOS = cph.ANOS
sinan = cph.carregar_sinan(ANOS)
sih = cph.carregar_sih(ANOS, cph.TODAS_UFS)

ler = lambda caminho: pd.read_csv(caminho, sep=";", encoding="utf-8-sig", dtype={"COMPETEN": str, "competencia": str})
ocup = ler("dados/resultados/hospitalar/ocupacao_uti_brasil.csv")
ocup_uf = ler("dados/resultados/hospitalar/ocupacao_uti_mensal_uf.csv")
leitos = ler("dados/resultados/leitos/leitos_uti_brasil.csv")

ocup["data"] = pd.to_datetime(ocup["COMPETEN"], format="%Y%m")
ocup["ano"], ocup["mes"] = ocup["data"].dt.year, ocup["data"].dt.month
ocup_uf["ano"] = ocup_uf["COMPETEN"].str[:4].astype(int)
leitos["data"] = pd.to_datetime(leitos["competencia"], format="%Y-%m")

print(f"\nSINAN: {len(sinan)} casos | SIH: {len(sih)} internacoes")
print(f"Ocupacao Brasil: {len(ocup)} meses ({ocup['COMPETEN'].iloc[0]} a {ocup['COMPETEN'].iloc[-1]})")
print(f"Ocupacao por UF: {len(ocup_uf)} linhas | Leitos: {len(leitos)} meses")

## Parte 3 — Análise da ocupação de UTI (Brasil, 2007–2026)
**Como a ocupação é calculada** (script `calcular_ocupacao_uti.py` do repositório): leitos ocupados =
dias de UTI do SIH ÷ dias do mês; o denominador são os leitos de UTI adulto + pediátrica **SUS** do CNES.
Parte dos dias de UTI vem sem tipo (`dias_nao_classificado`), então há uma **faixa** `ocupacao_min` –
`ocupacao_max`.

### [3.1] Resumo anual
Média anual dos leitos SUS, da ocupação (mín/máx) e dos leitos livres por 10 mil habitantes.

In [ ]:
# [3.1] Resumo anual
resumo_anual = ocup.groupby("ano").agg(
    meses=("COMPETEN", "count"),
    leitos_sus=("leitos_sus", "mean"),
    ocupacao_min=("ocupacao_min", "mean"),
    ocupacao_max=("ocupacao_max", "mean"),
    livres_min=("livres_min", "mean"),
    livres_max=("livres_max", "mean"),
    livres_por_10mil_min=("livres_por_10mil_min", "mean"),
    livres_por_10mil_max=("livres_por_10mil_max", "mean"),
)
resumo_anual

### [3.2] Séries mensais
Ocupação (faixa mín–máx) e leitos de UTI por 10 mil habitantes. O sombreado marca 2020–2022
(descritivo, sem atribuição de causa).

In [ ]:
# [3.2] Series mensais
def sombrear(ax):
    ax.axvspan(pd.Timestamp("2020-01-01"), pd.Timestamp("2022-12-31"), color="gray", alpha=.15, label="2020-2022")

fig, ax = plt.subplots(2, 1, figsize=(11, 7), sharex=True)
ax[0].fill_between(ocup["data"], ocup["ocupacao_min"], ocup["ocupacao_max"], alpha=.4, label="faixa mín-máx")
ax[0].plot(ocup["data"], ocup["ocupacao_max"], lw=.8)
sombrear(ax[0]); ax[0].set(ylabel="taxa de ocupação", title="Ocupação dos leitos de UTI SUS - Brasil"); ax[0].legend()

ax[1].plot(leitos["data"], leitos["uti_por_10mil_existentes"], label="existentes")
ax[1].plot(leitos["data"], leitos["uti_por_10mil_sus"], label="SUS")
sombrear(ax[1]); ax[1].set(ylabel="leitos / 10 mil hab.", xlabel="competência",
                           title="Leitos de UTI (adulto + pediátrica) por 10 mil habitantes"); ax[1].legend()
plt.tight_layout(); plt.show()

### [3.3] Sazonalidade
Média da ocupação por mês do ano. Duas curvas: todos os anos e sem 2020–2022. Como a ocupação tem
tendência (3.5), a média por mês mistura tendência e sazonalidade.

In [ ]:
# [3.3] Sazonalidade (media por mes do ano)
todos = ocup.groupby("mes")["ocupacao_max"].mean()
sem_pandemia = ocup[~ocup["ano"].between(2020, 2022)].groupby("mes")["ocupacao_max"].mean()
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(todos.index, todos, marker="o", label="todos os anos")
ax.plot(sem_pandemia.index, sem_pandemia, marker="s", label="sem 2020-2022")
ax.set(xlabel="mês", ylabel="ocupação máxima média", title="Sazonalidade da ocupação de UTI SUS")
ax.set_xticks(range(1, 13)); ax.legend(); plt.show()

### [3.4] Ocupação por UF
Ranking da ocupação máxima média (2007–2026) e mapa de calor UF × ano.

In [ ]:
# [3.4] Ocupacao por UF
por_uf = ocup_uf.groupby("UF")["ocupacao_max"].mean().sort_values(ascending=False)
mapa = ocup_uf.pivot_table(index="UF", columns="ano", values="ocupacao_max", aggfunc="mean").loc[por_uf.index]

fig, ax = plt.subplots(1, 2, figsize=(14, 7), gridspec_kw={"width_ratios": [1, 2.2]})
ax[0].barh(por_uf.index[::-1], por_uf.values[::-1]); ax[0].set(title="Ocupação máxima média por UF", xlabel="ocupação")
im = ax[1].imshow(mapa.values, aspect="auto", cmap="viridis")
ax[1].set_xticks(range(len(mapa.columns)), mapa.columns, rotation=90)
ax[1].set_yticks(range(len(mapa.index)), mapa.index)
ax[1].grid(False); ax[1].set(title="Ocupação máxima média por UF e ano")
fig.colorbar(im, ax=ax[1], label="ocupação"); plt.tight_layout(); plt.show()

### [3.5] Tendência por mínimos quadrados
Regressão linear (MQ ordinários) da ocupação mensal contra o tempo em anos:
`ocupação = a + b·t`. IC95 de `b` pela distribuição t com `n−2` graus de liberdade.

**Hipótese:** erros independentes. A série mensal tem autocorrelação, então o IC95 é **otimista**
(estreito demais). O ajuste descreve a tendência linear no período inteiro, sem extrapolação.

In [ ]:
# [3.5] Tendencia linear por minimos quadrados
t = (ocup["data"] - ocup["data"].iloc[0]).dt.days.to_numpy() / 365.25      # anos desde o inicio
linhas, ajustes = [], {}
for coluna in ["ocupacao_min", "ocupacao_max"]:
    y = ocup[coluna].to_numpy()
    (b, a), cov = np.polyfit(t, y, 1, cov=True)
    ep = np.sqrt(np.diag(cov))
    gl = len(t) - 2
    tcrit = stats.t.ppf(0.975, gl)
    pred = a + b * t
    r2 = 1 - ((y - pred) ** 2).sum() / ((y - y.mean()) ** 2).sum()
    ajustes[coluna] = (a, b)
    linhas.append({"serie": coluna, "inclinacao_por_ano": b, "ic95_inf": b - tcrit * ep[0],
                   "ic95_sup": b + tcrit * ep[0], "intercepto": a, "R2": r2, "n": len(t)})
tendencia = pd.DataFrame(linhas).set_index("serie")
display(tendencia)

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(ocup["data"], ocup["ocupacao_max"], lw=.8, label="ocupacao_max")
ax.plot(ocup["data"], ocup["ocupacao_min"], lw=.8, label="ocupacao_min")
for col, cor in [("ocupacao_max", "C0"), ("ocupacao_min", "C1")]:
    a, b = ajustes[col]; ax.plot(ocup["data"], a + b * t, "--", color=cor, label=f"tendência {col}")
ax.set(title="Tendência linear (mínimos quadrados)", ylabel="taxa de ocupação"); ax.legend(); plt.show()

## Parte 4 — Estimação por mínimos quadrados dos parâmetros hospitalares
**Parâmetros do diagrama** (os do script do repositório): I1 sai por `p_H·σ_I1` (→H) e `(1−p_H)·γ_I1` (→R);
H sai por `p_C·σ_H` (→C) e `(1−p_C)·γ_H` (→R); C sai por `d·σ_C` (→óbito) e `(1−d)·γ_C` (→R).

**Modelo e hipóteses:**

- O tempo em cada compartimento (I1, H, C) é **exponencial**. Como cada compartimento tem duas saídas
  concorrentes, o tempo de permanência tem a mesma distribuição qualquer que seja o destino, e as
  durações só identificam a **taxa total de saída**. Por isso σ e γ do mesmo compartimento são tomados
  **iguais a essa taxa** (σ = γ), e a divisão entre os destinos fica nas probabilidades p_H, p_C, d.
  γ_I1, γ_H e γ_C aparecem na tabela 4.4, com o mesmo valor de σ; a seção 4.7 testa essa igualdade.
- As durações são medidas em **dias inteiros** `D`. Se o tempo real é exponencial com taxa λ e `D` é a parte
  inteira, então `P(D ≤ d) = 1 − exp(−λ(d+1))`. Esta é a curva ajustada à distribuição acumulada empírica.
- Isso difere da estimativa do script, que usa `1/média` dos dias inteiros; a diferença é a discretização.
- As probabilidades (p_H, p_C, d, d_H) são ajustadas às proporções observadas **por ano**.

**Incerteza:** IC95 por *bootstrap* não paramétrico (reamostragem dos casos com reposição), percentis 2,5% e 97,5%.

### [4.1] Durações observadas
Mesmas regras de `estimar` (script do repositório):

- **I1→H** (SINAN): dias entre o início dos sintomas e a internação, nos internados, janela
  `DIAS_ATE_INTERNACAO` (0–30 dias).
- **H** (SIH): dias de enfermaria. Sem UTI: `DIAS_PERM`. Com UTI e óbito: `DIAS_PERM − UTI_DIAS`
  (supõe-se que o óbito ocorreu na UTI). Com UTI e alta não é separável (o SIH não registra a ordem).
- **C** (SIH): dias de UTI das internações com UTI.

In [ ]:
# [4.1] Duracoes observadas (dias)
hosp = sinan["ID_HOSPIT"]
dias_I1 = (sinan["DT_INTERNA"] - sinan["DT_SIN_PRI"]).dt.days
dias_I1 = dias_I1[(hosp == cph.HOSPIT_SIM) & dias_I1.between(*cph.DIAS_ATE_INTERNACAO)].to_numpy()

com_uti, obito = sih["UTI_DIAS"] > 0, sih["MORTE"] == 1
dias_H_c_uti = (sih.loc[com_uti & obito, "DIAS_PERM"] - sih.loc[com_uti & obito, "UTI_DIAS"]).clip(lower=0)
dias_H = pd.concat([sih.loc[~com_uti, "DIAS_PERM"], dias_H_c_uti]).to_numpy()
dias_C = sih.loc[com_uti, "UTI_DIAS"].to_numpy()

DURACOES = {"sigma_I1": dias_I1, "sigma_H": dias_H, "sigma_C": dias_C}
pd.DataFrame({k: pd.Series(v).describe() for k, v in DURACOES.items()}).T

### [4.2] Ajuste das taxas σ (taxa total de saída)
Para cada compartimento: distribuição acumulada empírica `F_emp(d)` por dia `d = 0, 1, …`, ajustada por
`scipy.optimize.least_squares` minimizando `Σ_d (F_emp(d) − (1 − exp(−λ(d+1))))²`.

**Escolha declarada:** o ajuste usa os dias até o quantil `QUANTIL_MAX` = 99% das durações (a cauda
longa tem poucos casos e a CDF ali já está perto de 1; incluí-la daria muitos pontos quase iguais a 1
e pouco informativos).
O bootstrap usa `B` = 500 reamostras e semente fixa.

In [ ]:
# [4.2] Ajuste MQ das taxas sigma
QUANTIL_MAX = 0.99
B = 500
rng = np.random.default_rng(42)

def cdf_empirica(dias, dmax):
    contagem = np.bincount(dias.astype(int), minlength=dmax + 1)[: dmax + 1]
    return np.cumsum(contagem) / len(dias)

def ajustar_taxa(dias, dmax=None):
    """Devolve (lambda, dmax, d, F_emp, rmse, R2)."""
    dias = np.asarray(dias)
    if dmax is None:
        dmax = int(np.quantile(dias, QUANTIL_MAX))
    d = np.arange(dmax + 1)
    f_emp = cdf_empirica(dias, dmax)
    residuo = lambda p: f_emp - (1 - np.exp(-p[0] * (d + 1)))
    lam0 = 1 / (dias.mean() + 0.5)
    res = optimize.least_squares(residuo, [lam0], bounds=(1e-6, np.inf))
    rmse = np.sqrt(np.mean(res.fun ** 2))
    r2 = 1 - (res.fun ** 2).sum() / ((f_emp - f_emp.mean()) ** 2).sum()
    return res.x[0], dmax, d, f_emp, rmse, r2

ajustes_taxa, boot_taxa = {}, {}
for nome, dias in DURACOES.items():
    lam, dmax, d, f_emp, rmse, r2 = ajustar_taxa(dias)
    ajustes_taxa[nome] = dict(lam=lam, dmax=dmax, d=d, f_emp=f_emp, rmse=rmse, r2=r2, n=len(dias))
    boot_taxa[nome] = np.array([ajustar_taxa(rng.choice(dias, len(dias)), dmax)[0] for _ in range(B)])
    print(f"{nome}: lambda = {lam:.4f}/dia  (n = {len(dias)}, dias ajustados 0..{dmax}, RMSE = {rmse:.4f}, R2 = {r2:.4f})")

### [4.3] Ajuste das probabilidades
Proporções observadas por ano `p_a` (com `n_a` casos):

| Parâmetro | Evento | Casos considerados |
|---|---|---|
| `p_H` | internado | casos do SINAN com `ID_HOSPIT` preenchido (sim/não), por ano de início dos sintomas |
| `p_C` | com dias de UTI | internações do SIH, por ano da internação |
| `d` | óbito | internações do SIH **com UTI** |
| `d_H` | óbito | internações do SIH **sem UTI** |

Dois ajustes de uma constante `p` (`least_squares`):

- **MQ ponderado** (peso `n_a`): minimiza `Σ n_a (p_a − p)²`. Com `p` constante isso equivale à proporção
  agregada; aparece aqui como verificação do ajuste.
- **MQ simples** (peso igual por ano): minimiza `Σ (p_a − p)²`, a média das proporções anuais. Dá o
  mesmo peso a anos com poucos casos, então é mais sensível a eles. Só entram anos com `n_a ≥ 1`.

O IC95 de ambos vem de bootstrap dos casos (mantendo o ano de cada caso).

In [ ]:
# [4.3] Ajuste MQ das probabilidades
ano_sinan = sinan["DT_SIN_PRI"].dt.year
info = hosp.isin([cph.HOSPIT_SIM, cph.HOSPIT_NAO]).to_numpy()
ano_sih = sih["DT_INTER"].dt.year

EVENTOS = {   # parametro: (ano de cada caso, evento 0/1)
    "p_H": (ano_sinan[info].to_numpy(), (hosp[info] == cph.HOSPIT_SIM).to_numpy().astype(int)),
    "p_C": (ano_sih.to_numpy(), com_uti.to_numpy().astype(int)),
    "d":   (ano_sih[com_uti].to_numpy(), obito[com_uti].to_numpy().astype(int)),
    "d_H": (ano_sih[~com_uti].to_numpy(), obito[~com_uti].to_numpy().astype(int)),
}
ANOS_IDX = np.arange(ANOS[0], ANOS[-1] + 1)

def proporcoes_anuais(anos, evento):
    n = np.bincount(anos - ANOS_IDX[0], minlength=len(ANOS_IDX))
    s = np.bincount(anos - ANOS_IDX[0], weights=evento, minlength=len(ANOS_IDX))
    ok = n > 0
    return s[ok] / n[ok], n[ok]

def ajustar_p(p_a, n_a, ponderado):
    w = np.sqrt(n_a) if ponderado else np.ones(len(n_a))       # raiz do peso, pois o resíduo é elevado ao quadrado
    res = optimize.least_squares(lambda p: w * (p_a - p[0]), [p_a.mean()], bounds=(0, 1))
    return res.x[0]

ajustes_p, boot_p = {}, {}
for nome, (anos, ev) in EVENTOS.items():
    p_a, n_a = proporcoes_anuais(anos, ev)
    ajustes_p[nome] = dict(p_ponderado=ajustar_p(p_a, n_a, True), p_simples=ajustar_p(p_a, n_a, False),
                           n=len(ev), anos=len(n_a), p_a=p_a, n_a=n_a)
    bs = []
    for _ in range(B):
        i = rng.integers(0, len(ev), len(ev))
        pa, na = proporcoes_anuais(anos[i], ev[i])
        bs.append((ajustar_p(pa, na, True), ajustar_p(pa, na, False)))
    boot_p[nome] = np.array(bs)
    a = ajustes_p[nome]
    print(f"{nome}: MQ ponderado = {a['p_ponderado']:.4f} | MQ simples = {a['p_simples']:.4f} | n = {a['n']} em {a['anos']} anos")

### [4.4] Tabela comparativa (σ, γ e probabilidades)
Estimativas por **mínimos quadrados** (este notebook) × estimativas **por média / proporção** do script do
repositório (`estimar`: taxa = 1/duração média; proporção agregada com IC de Wilson).

**γ_I1, γ_H, γ_C:** recebem o valor ajustado da taxa total de saída do próprio compartimento (igual ao σ
correspondente), pela hipótese descrita no início da Parte 4; é o mesmo critério do script. As linhas de γ
reaproveitam o ajuste e o IC95 de σ.

In [ ]:
# [4.4] Tabela comparativa MQ x script (sigma, gamma e probabilidades)
script = cph.estimar(sinan, sih).set_index("parametro")
pct = lambda x: np.percentile(x, [2.5, 97.5])

def linha_script(nome):
    return {"script_valor": script.loc[nome, "valor"], "script_ic95_inf": script.loc[nome, "ic95_inf"],
            "script_ic95_sup": script.loc[nome, "ic95_sup"]}

linhas = []
for nome, a in ajustes_taxa.items():
    lo, hi = pct(boot_taxa[nome])
    gama = nome.replace("sigma", "gamma")
    linhas.append({"parametro": nome, "metodo": "MQ (CDF exponencial discretizada)", "valor": a["lam"],
                   "ic95_inf": lo, "ic95_sup": hi, "n": a["n"], **linha_script(nome), "RMSE": a["rmse"], "R2": a["r2"]})
    linhas.append({"parametro": gama, "metodo": f"MQ, = {nome} (saidas concorrentes)", "valor": a["lam"],
                   "ic95_inf": lo, "ic95_sup": hi, "n": a["n"], **linha_script(gama), "RMSE": a["rmse"], "R2": a["r2"]})
for nome, a in ajustes_p.items():
    for k, (chave, rotulo) in enumerate([("p_ponderado", "MQ ponderado por n"), ("p_simples", "MQ simples (média dos anos)")]):
        lo, hi = pct(boot_p[nome][:, k])
        linhas.append({"parametro": nome, "metodo": rotulo, "valor": a[chave], "ic95_inf": lo, "ic95_sup": hi,
                       "n": a["n"], **linha_script(nome), "RMSE": np.nan, "R2": np.nan})
comparacao = pd.DataFrame(linhas)
comparacao

### [4.5] Qualidade do ajuste das taxas
Distribuição acumulada empírica × ajustada, e resíduos por dia. Resíduos sistemáticos indicam que a
hipótese exponencial não descreve bem aquela permanência.

In [ ]:
# [4.5] CDF empirica x ajustada e residuos
fig, ax = plt.subplots(2, 3, figsize=(14, 7), sharex="col")
for j, (nome, a) in enumerate(ajustes_taxa.items()):
    ajuste = 1 - np.exp(-a["lam"] * (a["d"] + 1))
    ax[0, j].step(a["d"], a["f_emp"], where="post", label="empírica")
    ax[0, j].plot(a["d"], ajuste, "--", label=f"ajuste (λ = {a['lam']:.3f}/dia)")
    ax[0, j].set(title=f"{nome} (n = {a['n']})", ylabel="P(D ≤ d)"); ax[0, j].legend()
    ax[1, j].bar(a["d"], a["f_emp"] - ajuste); ax[1, j].axhline(0, color="k", lw=.8)
    ax[1, j].set(xlabel="dias d", ylabel="resíduo")
plt.tight_layout(); plt.show()

### [4.6] Qualidade do ajuste das probabilidades
Proporção anual `p_a` (tamanho do ponto ∝ `n_a`) e o valor ajustado. Anos com poucos casos variam muito.

In [ ]:
# [4.6] Proporcoes anuais x ajuste
fig, ax = plt.subplots(1, 4, figsize=(16, 3.8))
for axi, (nome, a) in zip(ax, ajustes_p.items()):
    anos_ok = [y for y in ANOS_IDX if (EVENTOS[nome][0] == y).any()]
    axi.scatter(anos_ok, a["p_a"], s=10 + 300 * a["n_a"] / a["n_a"].max(), alpha=.6)
    axi.axhline(a["p_ponderado"], color="C1", label="MQ ponderado")
    axi.axhline(a["p_simples"], color="C2", ls="--", label="MQ simples")
    axi.set(title=nome, xlabel="ano")
ax[0].legend(); plt.tight_layout(); plt.show()

### [4.7] Verificação: taxas por destino (σ × γ)
Se as saídas de um compartimento são exponenciais concorrentes, o tempo de permanência tem a **mesma
taxa** qualquer que seja o destino, e σ = γ é coerente. Aqui cada taxa é ajustada **separadamente por
destino observado** (mesmo método de 4.2), para conferir:

| Compartimento | Destino | Duração usada |
|---|---|---|
| H | →R (γ_H) | `DIAS_PERM` das internações **sem UTI e com alta** |
| H | →C (σ_H) | `DIAS_PERM − UTI_DIAS` das internações com UTI e **óbito** (único caso separável) |
| C | →R (γ_C) | `UTI_DIAS` das internações com UTI e **alta** |
| C | →óbito (σ_C) | `UTI_DIAS` das internações com UTI e **óbito** |

**Limite:** os destinos raros têm poucos casos (9 em H→C e 9 em C→óbito, segundo o script), então esses
IC95 são largos. Intervalos que se sobrepõem **não confirmam** a igualdade; apenas não a rejeitam.
I1 não entra: o SINAN não registra a data de cura, então o tempo I1→R não é observável.

In [ ]:
# [4.7] Taxas por destino (verificacao de sigma = gamma)
por_destino = {
    "gamma_H (H->R)":   sih.loc[~com_uti & ~obito, "DIAS_PERM"].to_numpy(),
    "sigma_H (H->C)":   dias_H_c_uti.to_numpy(),
    "gamma_C (C->R)":   sih.loc[com_uti & ~obito, "UTI_DIAS"].to_numpy(),
    "sigma_C (C->D)":   sih.loc[com_uti & obito, "UTI_DIAS"].to_numpy(),
}
linhas = []
for nome, dias in por_destino.items():
    lam, dmax, *_ , rmse, r2 = ajustar_taxa(dias)
    boot = [ajustar_taxa(rng.choice(dias, len(dias)), dmax)[0] for _ in range(B)]
    lo, hi = pct(boot)
    linhas.append({"taxa": nome, "n": len(dias), "dias_medios": dias.mean(), "lambda_MQ": lam,
                   "ic95_inf": lo, "ic95_sup": hi, "RMSE": rmse})
destinos = pd.DataFrame(linhas).set_index("taxa")
destinos

### [4.8] Limitações
- Só **114 internações** com UTI no SIH de 2007 a 2026: `σ_C`, `γ_C` e `d` têm IC95 largo, e as taxas por
  destino de 4.7 se apoiam em poucos casos (H→C e C→óbito).
- O SINAN não registra a data de cura, então o tempo I1→R **não é observável**; `γ_I1` é igual a `σ_I1`
  por hipótese e não pode ser verificado como em 4.7.
- Em H com UTI e alta, o SIH não informa quanto da permanência foi antes ou depois da UTI; esses casos
  ficam fora de `σ_H`.
- A hipótese de tempo exponencial pode não valer; veja os resíduos em 4.5.
- `ID_HOSPIT` em branco: `p_H` usa só os casos informados (o script oferece pós-estratificação e limite
  inferior `p_H_minimo`, que não entram aqui).
- Os dados são de **sarampo**; os parâmetros não valem para COVID-19 sem outra fonte.

## Parte 5 — Exportar resultados
### [5.1] Salvar e baixar
Grava `comparacao_parametros_mq.csv` (tabela 4.4), `taxas_por_destino_mq.csv` (4.7),
`resumo_anual_ocupacao_uti.csv` (3.1) e `tendencia_ocupacao_uti.csv` (3.5), e baixa os quatro (no Colab).

In [ ]:
# [5.1] Exportar
saidas = {"comparacao_parametros_mq.csv": comparacao,
          "taxas_por_destino_mq.csv": destinos.reset_index(),
          "resumo_anual_ocupacao_uti.csv": resumo_anual.reset_index(),
          "tendencia_ocupacao_uti.csv": tendencia.reset_index()}
for nome, df in saidas.items():
    df.to_csv(nome, sep=";", index=False, encoding="utf-8-sig")
    print("gravado:", Path(nome).resolve())
try:
    from google.colab import files
    for nome in saidas:
        files.download(nome)
except ImportError:
    pass